# 06 · RMSF — flutuação por resíduo e fator B

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** Que regiões da xilanase são flexíveis, e o perfil de flexibilidade da simulação tem a mesma forma do fator B cristalográfico (1XNB)?

$\mathrm{RMSF}_i = \sqrt{\langle |\mathbf{r}_i(t) - \bar{\mathbf{r}}_i|^2 \rangle_t}$ na
produção, cada réplica alinhada à **própria estrutura média** (alinhar ao frame 0 infla o RMSF de
forma sistemática: a média é o mínimo de mínimos quadrados). Não é série temporal: a incerteza de
cada resíduo vem da dispersão **entre réplicas** (ou entre blocos, com uma réplica só — limite
inferior).

**Validação com dado experimental:** $B = \tfrac{8\pi^2}{3}\langle u^2 \rangle$, com
$\langle u^2 \rangle$ = **média do MSF** entre réplicas (o quadrado da média do RMSF subestima B,
pela desigualdade de Jensen). Compare a **forma** (correlação), não o valor: o B cristalográfico
inclui desordem estática, empacotamento e refinamento, e a temperatura de coleta raramente é a
da simulação.

> **Pré-processamento** (a PBC é tratada no GROMACS, não em Python):
> ```bash
> gmx trjconv -s md_300ns.tpr -f md_300ns.xtc -o md_whole.xtc -pbc whole
> gmx trjconv -s md_300ns.tpr -f md_whole.xtc -o md_300ns_mol.xtc -pbc mol -center -ur compact
> ```

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mdtraj as md
import MDAnalysis as mda
from scipy.signal import find_peaks
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
    "MDTraj": md.__version__,
    "MDAnalysis": mda.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

In [ ]:
# ── Onde estão as simulações ────────────────────────────────────────────────
PASTA_BASE = Path("..")             # pasta que contém rep1/, rep2/, ...   <- AJUSTE
SISTEMA = "xilanase_1XNB"           # nome da pasta de resultados
REPLICAS = [1, 2, 3, 4, 5]          # uma réplica só também funciona: [1]
PREFIXO_MD = "md_300ns"             # -deffnm da produção


def pasta_replica(n):
    """Pasta da réplica n."""
    return PASTA_BASE / f"rep{n}"

ARQUIVO_TOP = f"{PREFIXO_MD}.gro"            # sistema inteiro
ARQUIVO_TRAJ = f"{PREFIXO_MD}_mol.xtc"        # sistema inteiro, -pbc mol -center
ARQUIVO_TPR = f"{PREFIXO_MD}.tpr"            # elementos pelo nº atômico (MDAnalysis >= 2.11); None = pelo nome
ARQUIVOS = {"topologia": ARQUIVO_TOP, "trajetória": ARQUIVO_TRAJ, **({"tpr": ARQUIVO_TPR} if ARQUIVO_TPR else {})}

# ── Amostragem ──────────────────────────────────────────────────────────────
DT_ANALISE_PS = 100.0     # intervalo entre frames analisados
DESCARTE_NS = 0.0         # equilibração descartada no início de cada réplica (decida no 02)
NIVEL_CONFIANCA = 0.95    # intervalo de confiança (t de Student)

# ── Análise ─────────────────────────────────────────────────────────────────
SELECAO_CARGA = "protein"            # proteína inteira (o DSSP precisa do esqueleto completo)
SELECAO_AJUSTE = "name CA"
SELECAO_RMSF = "name CA"             # um átomo por resíduo
ITERACOES_MEDIA = 3                  # refinamentos da estrutura média
N_BLOCOS = 5                         # barra de erro por blocos quando há UMA réplica
CALCULAR_DSSP = True                 # faixas de hélice/folha no gráfico
MULTIPLICADOR_DESVIO = 1.5           # triagem de picos: média + N·σ entre resíduos
DISTANCIA_PICOS = 5                  # separação mínima entre picos anotados (resíduos)
SEPARACAO_ROTULOS = 12               # picos mais próximos que isso têm os rótulos abertos para lados opostos

# ── Fator B cristalográfico (None = sem comparação) ─────────────────────────
PDB_BFATOR = PASTA_BASE / "protein" / "1XNB.pdb"
CADEIA_BFATOR = None                 # ex.: "A"; None aceita qualquer cadeia
DESLOCAMENTO_RESIDUO = 0             # somado ao nº do resíduo do PDB antes de casar
RESIDUOS_EXCLUIDOS = 3               # aparados em cada extremidade na 2ª correlação

# ── Saída ───────────────────────────────────────────────────────────────────
DIR_SAIDA = Path("resultados") / SISTEMA / "06_rmsf"
PREFIXO = "rmsf"
TITULO = "06 · RMSF e fator B"

In [ ]:
linhas = []
for n in REPLICAS:
    for papel, nome in ARQUIVOS.items():
        caminho = pasta_replica(n) / nome
        linhas.append({"replica": f"rep{n}", "arquivo": papel, "caminho": str(caminho),
                       "existe": caminho.exists(),
                       "tamanho_MB": round(caminho.stat().st_size / 1024 ** 2, 1) if caminho.exists() else np.nan})
df_arquivos = pd.DataFrame(linhas)
display(df_arquivos)
if not df_arquivos["existe"].all():
    raise FileNotFoundError("Arquivos ausentes (veja a tabela acima). Ajuste RAIZ/SISTEMA/REPLICAS.")

In [ ]:
MAPA_RESNAMES = {"HIE": "HIS", "HID": "HIS", "HIP": "HIS", "HSD": "HIS", "HSE": "HIS",
                 "HSP": "HIS", "CYX": "CYS", "CYM": "CYS", "LYN": "LYS", "ASH": "ASP",
                 "GLH": "GLU"}


def tpr_de(n):
    """Caminho do .tpr da réplica n (None se ARQUIVO_TPR = None)."""
    return pasta_replica(n) / ARQUIVO_TPR if ARQUIVO_TPR else None


def elementos_do_tpr(tpr, top, arquivo):
    """Elemento de cada átomo da topologia, pelo número atômico gravado no .tpr.

    O .gro pode ser o sistema inteiro ou um subconjunto na MESMA ordem (o _prot.gro).
    O casamento é resíduo a resíduo, pela sequência de nomes de átomos: o MDAnalysis
    renumera os resíduos do .tpr a partir de 1, então o nº do resíduo (Leu4...) não serve.
    """
    u = mda.Universe(str(tpr))
    if not hasattr(u.atoms, "elements"):
        raise ValueError("o .tpr não trouxe números atômicos")
    if str(arquivo).endswith(".gro"):            # nomes crus (o MDTraj renomeia OW -> O etc.)
        linhas = Path(arquivo).read_text().splitlines()[2:2 + top.n_atoms]
        nomes = [l[10:15].strip() for l in linhas]
    else:
        nomes = [a.name for a in top.atoms]
    cortes = np.flatnonzero(np.diff(u.atoms.resindices)) + 1
    residuos_tpr = list(zip(np.split(u.atoms.names, cortes), np.split(u.atoms.elements, cortes)))
    elementos, j = [], 0
    for residuo in top.residues:
        alvo = [nomes[a.index] for a in residuo.atoms]
        while j < len(residuos_tpr) and list(residuos_tpr[j][0]) != alvo:
            j += 1
        if j == len(residuos_tpr):
            raise ValueError(f"{residuo} não casa com o .tpr (o .gro não é um subconjunto ordenado dele)")
        elementos += list(residuos_tpr[j][1])
        j += 1
    return elementos


def ler_topologia(arquivo, tpr=None):
    """Topologia MDTraj de um .gro/.pdb, com os elementos certos.

    * Com `tpr` (MDAnalysis >= 2.11 lê o .tpr do GROMACS 2026): o elemento vem do número
      atômico gravado no .tpr — a mesma informação que o GROMACS usa.
    * Sem `tpr` (ou se a leitura falhar): o MDTraj deduz o elemento pelo nome do átomo, e o
      FE do heme viraria flúor (19 u em vez de 55,85 u); ele é corrigido e um aviso é impresso.
    * Resíduos com nome de protonação (HIE, CYM...) recebem o nome padrão, para que
      `protein` e `resname` funcionem.
    """
    top = md.load_topology(str(arquivo))
    for residuo in top.residues:
        residuo.name = MAPA_RESNAMES.get(residuo.name.upper(), residuo.name)
    if tpr is not None and Path(tpr).exists():
        try:
            elementos = elementos_do_tpr(tpr, top, arquivo)
            for atomo, simbolo in zip(top.atoms, elementos):
                atomo.element = md.element.get_by_symbol(simbolo.capitalize()) if simbolo else md.element.virtual
            return top
        except Exception as erro:
            print(f"[aviso] .tpr não usado ({erro}); elementos deduzidos pelo nome do átomo.")
    for atomo in top.atoms:
        if atomo.residue.name == "HEM" and atomo.name.upper() == "FE":
            atomo.element = md.element.iron
    return top


def selecionar(top, consulta, n=None):
    """top.select com erro explícito se vazio (ou se não tiver n átomos)."""
    idx = top.select(consulta)
    if len(idx) == 0:
        raise ValueError(f"Seleção vazia: '{consulta}' (no MDTraj o nº do resíduo é `resSeq`).")
    if n is not None and len(idx) != n:
        raise ValueError(f"'{consulta}' casou com {len(idx)} átomos (esperado {n}).")
    return idx


def intervalo_do_arquivo_ps(xtc, top):
    """Δt entre frames gravados no .xtc (lê só os 2 primeiros frames)."""
    for lote in md.iterload(str(xtc), top=top, chunk=2, atom_indices=[0]):
        return float(np.diff(lote.time)[0])


def estride(xtc, top, dt_alvo_ps):
    """Passo de leitura que leva o Δt do arquivo ao Δt de análise."""
    dt = intervalo_do_arquivo_ps(xtc, top)
    passo = max(1, int(round(dt_alvo_ps / dt)))
    if dt_alvo_ps < dt:
        print(f"[aviso] Δt pedido {dt_alvo_ps:g} ps é menor que o do arquivo ({dt:g} ps).")
    elif abs(passo * dt - dt_alvo_ps) > 1e-6 * dt_alvo_ps:
        print(f"[aviso] Δt {dt_alvo_ps:g} ps não é múltiplo de {dt:g} ps; usando {passo * dt:g} ps.")
    return passo, dt


def producao(t_ns):
    """Máscara dos frames de produção (depois de DESCARTE_NS)."""
    t_ns = np.asarray(t_ns, dtype=float)
    mascara = t_ns >= t_ns[0] + DESCARTE_NS
    if not mascara.any():
        raise ValueError("DESCARTE_NS é maior que a trajetória.")
    return mascara


def carregar(n, atomos=None, dt_ps=None):
    """Lê a réplica n (só os átomos `atomos`, índices da topologia) -> (traj, t_ns)."""
    xtc, top = pasta_replica(n) / ARQUIVO_TRAJ, ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    passo, dt = estride(xtc, top, dt_ps or DT_ANALISE_PS)
    traj = md.load(str(xtc), top=top, stride=passo,
                   atom_indices=None if atomos is None else np.unique(atomos))
    t_ns = np.asarray(traj.time, dtype=float) / 1000.0
    if not np.all(np.diff(t_ns) > 0):
        raise ValueError(f"rep{n}: tempos não crescentes (use `gmx trjcat -cat`).")
    return traj, t_ns



top = ler_topologia(pasta_replica(REPLICAS[0]) / ARQUIVO_TOP, tpr_de(REPLICAS[0]))
trajs = {f"rep{n}": carregar(n, selecionar(top, SELECAO_CARGA)) for n in REPLICAS}
top_c = next(iter(trajs.values()))[0].topology
idx_ajuste, idx_rmsf = selecionar(top_c, SELECAO_AJUSTE), selecionar(top_c, SELECAO_RMSF)
residuos = np.array([top_c.atom(int(i)).residue.resSeq for i in idx_rmsf])
nomes = np.array([top_c.atom(int(i)).residue.name.capitalize() for i in idx_rmsf])
if len(set(residuos)) != len(residuos):
    print("[aviso] SELECAO_RMSF tem mais de um átomo por resíduo.")
print(f"{idx_rmsf.size} posições | resíduos {residuos.min()}–{residuos.max()}")
df_trajetorias = pd.DataFrame([{
    "replica": rep, "frames": tr.n_frames, "atomos": tr.n_atoms,
    "dt_ps": round(1000 * float(np.median(np.diff(t))), 3),
    "inicio_ns": t[0], "fim_ns": t[-1],
    "frames_producao": int(producao(t).sum()),
    "memoria_MB": round(tr.xyz.nbytes / 1024 ** 2, 1)}
    for rep, (tr, t) in trajs.items()])
df_trajetorias

In [ ]:
df_cristal = pd.DataFrame()
if PDB_BFATOR and Path(PDB_BFATOR).exists():
    linhas_pdb = Path(PDB_BFATOR).read_text().splitlines()
    temperatura = [l[45:].strip() for l in linhas_pdb if l.startswith("REMARK 200  TEMPERATURE")]
    # Colunas fixas do formato PDB (split() quebra quando campos encostam); 1ª conformação
    df_cristal = pd.DataFrame([{"residuo": int(l[22:26]) + DESLOCAMENTO_RESIDUO,
                                "nome_pdb": MAPA_RESNAMES.get(l[17:20].strip(), l[17:20].strip()).capitalize(),
                                "b_cristal_A2": float(l[60:66])}
                               for l in linhas_pdb if l.startswith("ATOM") and l[12:16] == " CA "
                               and l[16] in (" ", "A") and (CADEIA_BFATOR is None or l[21] == CADEIA_BFATOR)]
                              ).drop_duplicates("residuo")
    print(f"{PDB_BFATOR}: {len(df_cristal)} Cα | temperatura de coleta: "
          f"{temperatura[0] if temperatura else 'não registrada'}")
    display(df_cristal.head())
else:
    print("[fator B] PDB_BFATOR ausente — comparação pulada.")

## 3 · Cálculos

O RMSF é calculado explicitamente (o `md.rmsf` centra as conformações no lugar). Perfis por
bloco da produção alimentam a barra de erro quando há uma réplica só. DSSP simplificado
(H, E, C) somado sobre **todos** os frames de **todas** as réplicas.

In [ ]:
def alinhar_a_media(traj, idx, iteracoes=3):
    """Alinha à estrutura média, refinada iterativamente; devolve (alinhada, média)."""
    trabalho = traj[:]
    ref = trabalho[0]
    for _ in range(iteracoes):
        trabalho.superpose(ref, atom_indices=idx, ref_atom_indices=idx)
        ref = trabalho[0]
        ref.xyz = trabalho.xyz.mean(axis=0, keepdims=True)
    trabalho.superpose(ref, atom_indices=idx, ref_atom_indices=idx)
    return trabalho, ref


def msf(xyz, ref):
    return ((xyz.astype(np.float64) - ref) ** 2).sum(axis=2).mean(axis=0)


quadros, blocos, contagem_dssp = [], [], 0
for rep, (traj, t_ns) in trajs.items():
    alinhada, media = alinhar_a_media(traj[producao(t_ns)], idx_ajuste, ITERACOES_MEDIA)
    xyz, ref = alinhada.xyz[:, idx_rmsf], media.xyz[0, idx_rmsf]
    m = msf(xyz, ref)
    quadros.append(pd.DataFrame({"replica": rep, "residuo": residuos, "nome": nomes,
                                 "rmsf_nm": np.sqrt(m), "msf_nm2": m}))
    for b, fatia in enumerate(np.array_split(np.arange(xyz.shape[0]), N_BLOCOS), start=1):
        blocos.append(pd.DataFrame({"replica": rep, "bloco": b, "residuo": residuos,
                                    "rmsf_nm": np.sqrt(msf(xyz[fatia], ref))}))
    if CALCULAR_DSSP:
        codigos = md.compute_dssp(alinhada, simplified=True)
        contagem_dssp = contagem_dssp + pd.DataFrame({c: (codigos == c).sum(axis=0) for c in ("H", "E", "C")},
                                                     index=[r.resSeq for r in alinhada.topology.residues])
df_rmsf = pd.concat(quadros, ignore_index=True)
df_blocos = pd.concat(blocos, ignore_index=True)
df_dssp = pd.DataFrame()
if CALCULAR_DSSP:
    fracoes = contagem_dssp.div(contagem_dssp.sum(axis=1), axis=0)
    df_dssp = pd.DataFrame({"residuo": fracoes.index, "frac_helice": fracoes["H"].to_numpy(),
                            "frac_folha": fracoes["E"].to_numpy(), "frac_coil": fracoes["C"].to_numpy(),
                            "dssp": fracoes.idxmax(axis=1).to_numpy()})
    print("Estrutura secundária (consenso):", df_dssp["dssp"].value_counts().to_dict())
df_rmsf.groupby("replica", sort=False)["rmsf_nm"].describe()

## 4 · Estatística

* **Por resíduo:** média ± SEM entre réplicas (com 1 réplica, entre `N_BLOCOS` blocos), IC t de
  Student. Aqui não há autocorrelação a corrigir: cada valor já é uma média sobre a produção.
* **σ entre resíduos × SEM por resíduo:** a primeira é o sinal (o perfil), a segunda a
  incerteza. Picos só são anotados se superam o limiar **por mais que a própria incerteza**.
* **Fator B:** Pearson (linear) e Spearman (monotônica), com e sem as extremidades, e dois
  ajustes lineares de B$_{cristal}$ contra B$_{DM}$. Um teste de identidade dos resíduos pega
  deslocamento de numeração.
  * **Fator de escala** (o "× 0,7" da figura): mínimos quadrados **pela origem**,
    $s = \sum B_{DM} B_{cristal} / \sum B_{DM}^2$, de modo que $B_{cristal} \approx s\,B_{DM}$. Serve só
    para pôr os dois perfis no mesmo eixo e comparar a **forma**; não tem significado físico. Como
    cada resíduo pesa $B_{DM}^2$, os poucos resíduos muito móveis na DM (alças, extremidades)
    dominam o ajuste e puxam $s$ para baixo de 1.
  * **Regressão com intercepto**, $B_{cristal} = a + b\,B_{DM}$: $a$ estima a parte do B
    cristalográfico que a DM em solução não tem (desordem estática, vibração da rede, movimento de
    corpo rígido do cristal, removido pelo alinhamento na DM), e $b$ a proporção da flutuação
    dinâmica que o cristal reproduz.

In [ ]:
def ineficiencia_estatistica(x, t_minimo=3):
    """g = 1 + 2·τ_int (em frames) — Chodera et al., JCTC 2007, 3, 26.

    ACF não enviesada (FFT), somada até o 1º C(t) <= 0 depois de t_minimo, com o
    peso (1 − t/N) aplicado UMA vez. Igual a
    pymbar.timeseries.statistical_inefficiency(fast=False).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 3 or np.allclose(x, x[0]):
        return 1.0
    d = x - x.mean()
    var = float(d @ d) / n
    tamanho = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(d, tamanho)
    acf = np.fft.irfft(f * np.conjugate(f), tamanho)[:n] / ((n - np.arange(n)) * var)
    g = 1.0
    for t in range(1, n - 1):
        if acf[t] <= 0.0 and t > t_minimo:
            break
        g += 2.0 * acf[t] * (1.0 - t / n)
    return float(max(g, 1.0))


def t_student(graus_liberdade):
    return stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=max(graus_liberdade, 1))


def resumo_replica(y, dt_ns, observavel, replica, unidade=""):
    """Média, DP, g, N_ef e SEM corrigido de UMA série (uma réplica)."""
    y = np.asarray(y, dtype=float)
    g = ineficiencia_estatistica(y)
    n_ef = max(y.size / g, 2.0)
    dp = float(np.std(y, ddof=1))
    sem = dp / np.sqrt(n_ef)
    media = float(np.mean(y))
    return {"observavel": observavel, "replica": replica, "unidade": unidade,
            "n": y.size, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(n_ef - 1) * sem,
            "ic_superior": media + t_student(n_ef - 1) * sem,
            "minimo": float(y.min()), "maximo": float(y.max()),
            "g": g, "tau_int_ns": (g - 1.0) / 2.0 * dt_ns, "n_efetivo": n_ef}


def entre_replicas(medias, observavel, unidade=""):
    """A réplica é a unidade estatística: média ± SEM das médias das réplicas."""
    medias = np.asarray(medias, dtype=float)
    k = medias.size
    media = float(medias.mean())
    dp = float(np.std(medias, ddof=1)) if k > 1 else np.nan
    sem = dp / np.sqrt(k) if k > 1 else np.nan
    return {"observavel": observavel, "replica": "entre_replicas", "unidade": unidade,
            "n": k, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(k - 1) * sem if k > 1 else np.nan,
            "ic_superior": media + t_student(k - 1) * sem if k > 1 else np.nan,
            "minimo": float(medias.min()), "maximo": float(medias.max()),
            "g": np.nan, "tau_int_ns": np.nan, "n_efetivo": float(k)}


def estatisticas_series(df_series, unidades):
    """Por réplica (só produção) + linha 'entre_replicas' quando há 2 ou mais."""
    linhas = []
    prod = df_series[df_series["fase"] == "producao"]
    for obs, bloco in prod.groupby("observavel", sort=False):
        por_rep = [resumo_replica(g["valor"], float(np.median(np.diff(g["tempo_ns"]))),
                                  obs, rep, unidades.get(obs, ""))
                   for rep, g in bloco.groupby("replica", sort=False)]
        linhas += por_rep
        if len(por_rep) > 1:
            linhas.append(entre_replicas([l["media"] for l in por_rep], obs, unidades.get(obs, "")))
    return pd.DataFrame(linhas)


def referencia(df_stats, observavel):
    """Linha 'entre_replicas' do observável (ou a da única réplica)."""
    bloco = df_stats[df_stats["observavel"] == observavel]
    entre = bloco[bloco["replica"] == "entre_replicas"]
    return (entre if len(entre) else bloco).iloc[0]


def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()

In [ ]:
tab = df_rmsf.pivot(index="residuo", columns="replica", values="rmsf_nm")
tab_msf = df_rmsf.pivot(index="residuo", columns="replica", values="msf_nm2")
k = tab.shape[1]
if k > 1:
    media, dp, gl, fonte_erro = tab.mean(axis=1), tab.std(axis=1, ddof=1), k - 1, f"entre {k} réplicas"
    n_amostras = k
else:
    bl = df_blocos.pivot(index="residuo", columns="bloco", values="rmsf_nm")
    media, dp, gl, fonte_erro = tab.iloc[:, 0], bl.std(axis=1, ddof=1), N_BLOCOS - 1, f"entre {N_BLOCOS} blocos (1 réplica)"
    n_amostras = N_BLOCOS
sem = dp / np.sqrt(n_amostras)
df_perfil = pd.DataFrame({"residuo": tab.index, "nome": pd.Series(nomes, index=residuos).reindex(tab.index).to_numpy(),
                          "rmsf_medio_nm": media.to_numpy(), "desvio_padrao_nm": dp.to_numpy(), "sem_nm": sem.to_numpy(),
                          "ic_inferior_nm": (media - t_student(gl) * sem).to_numpy(),
                          "ic_superior_nm": (media + t_student(gl) * sem).to_numpy(),
                          "b_dm_A2": (8 * np.pi ** 2 / 3 * 100 * tab_msf.mean(axis=1)).to_numpy()})   # nm² -> Å²
if len(df_dssp):
    df_perfil = df_perfil.merge(df_dssp, on="residuo", how="left")

media_res, sigma_res = float(df_perfil["rmsf_medio_nm"].mean()), float(df_perfil["rmsf_medio_nm"].std(ddof=1))
limite = media_res + MULTIPLICADOR_DESVIO * sigma_res
picos, _ = find_peaks(df_perfil["rmsf_medio_nm"], height=limite, distance=DISTANCIA_PICOS)
picos = np.array([i for i in picos if df_perfil["rmsf_medio_nm"][i] - np.nan_to_num(df_perfil["sem_nm"][i]) > limite], dtype=int)
rotulos_picos = [f"{df_perfil['nome'][i]}{df_perfil['residuo'][i]}" for i in picos]

df_stats = pd.DataFrame([{"observavel": "RMSF médio da proteína", "replica": rep, "unidade": "nm",
                          "media": float(v), "sem": np.nan} for rep, v in tab.mean(axis=0).items()]
                        + ([entre_replicas(tab.mean(axis=0), "RMSF médio da proteína", "nm")] if k > 1 else []))
df_resumo = pd.DataFrame([{"fonte_do_erro": fonte_erro, "media_entre_residuos_nm": media_res,
                           "sigma_entre_residuos_nm": sigma_res, "sem_mediano_nm": float(df_perfil["sem_nm"].median()),
                           "razao_sigma_sem": sigma_res / float(df_perfil["sem_nm"].median()),
                           "limite_triagem_nm": limite, "n_picos": len(picos),
                           "picos": ", ".join(rotulos_picos) or "—"}])

df_comparacao, df_correlacao = pd.DataFrame(), pd.DataFrame()
if not df_cristal.empty:
    df_comparacao = df_perfil.merge(df_cristal, on="residuo", how="inner")
    if df_comparacao.empty:
        raise ValueError(f"Nenhum resíduo casou: DM {df_perfil['residuo'].min()}–{df_perfil['residuo'].max()}, "
                         f"PDB {df_cristal['residuo'].min()}–{df_cristal['residuo'].max()} — ajuste DESLOCAMENTO_RESIDUO.")
    divergentes = int((df_comparacao["nome"].str.upper() != df_comparacao["nome_pdb"].str.upper()).sum())
    if divergentes:
        print(f"[fator B] {divergentes}/{len(df_comparacao)} resíduos casam por número mas divergem em identidade.")
    linhas = []
    aparado = df_comparacao.iloc[RESIDUOS_EXCLUIDOS:len(df_comparacao) - RESIDUOS_EXCLUIDOS]
    for conjunto, q in (("todos", df_comparacao), (f"sem {RESIDUOS_EXCLUIDOS} por extremidade", aparado)):
        x, y = q["b_dm_A2"].to_numpy(), q["b_cristal_A2"].to_numpy()
        r, p = stats.pearsonr(x, y)
        rho, p_s = stats.spearmanr(x, y)
        regressao = stats.linregress(x, y)
        linhas.append({"conjunto": conjunto, "n": len(q), "pearson_r": r, "pearson_p": p, "spearman_rho": rho,
                       "spearman_p": p_s,
                       "escala_exp_sobre_dm": float(x @ y / (x @ x)),         # mínimos quadrados pela origem
                       "intercepto_A2": regressao.intercept, "u_intercepto_A2": regressao.intercept_stderr,
                       "inclinacao": regressao.slope, "u_inclinacao": regressao.stderr})
    df_correlacao = pd.DataFrame(linhas)
    display(df_correlacao)
if len(REPLICAS) == 1:
    print("[1 réplica] Não há linha 'entre_replicas': o erro é o SEM da própria série, "
          "corrigido pela ineficiência estatística (subestima a incerteza de observáveis lentos).")
display(df_resumo.T.rename(columns={0: "valor"}))
df_stats

## 5 · Gráfico (matplotlib)

**Figura 1** — perfil médio (réplicas ao fundo), faixas de hélice e folha (consenso do DSSP) e
picos da triagem. **Figura 2** — fator B: perfis no mesmo eixo, com o B da DM multiplicado pelo
fator de escala (compara a forma), e dispersão com a reta de escala.

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Arial"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 10                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

### Figura 1 — perfil de RMSF

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 6.3, 3.5
COR_PERFIL = COR_MEDIA            # média entre réplicas
COR_HELICE, COR_FOLHA, OPACIDADE_SS = "#5b8ff9", "#f6bd16", 0.18
COR_PICOS = "#bc5090"
MOSTRAR_REPLICAS = True
LIMITES_Y = None                 # None = (0, 1,6 × máximo)

# ── Gráfico ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(LARGURA, ALTURA), layout="constrained")
x, y = df_perfil["residuo"].to_numpy(), df_perfil["rmsf_medio_nm"].to_numpy()
if "dssp" in df_perfil:
    for r, ss in zip(x, df_perfil["dssp"]):
        if ss in ("H", "E"):
            ax.axvspan(r - 0.5, r + 0.5, color=COR_HELICE if ss == "H" else COR_FOLHA, alpha=OPACIDADE_SS, lw=0)
    ax.fill_between([], [], color=COR_HELICE, alpha=0.45, label="hélice")
    ax.fill_between([], [], color=COR_FOLHA, alpha=0.45, label="folha-β")
if MOSTRAR_REPLICAS and k > 1:
    for i, (rep, g) in enumerate(df_rmsf.groupby("replica", sort=False)):
        ax.plot(g["residuo"], g["rmsf_nm"], color=CORES[i % len(CORES)], lw=0.5, alpha=0.7, label=rep)
ep = df_perfil["sem_nm"].to_numpy()
ax.fill_between(x, y - ep, y + ep, color=COR_PERFIL, alpha=0.25, lw=0)
ax.plot(x, y, color=COR_PERFIL, lw=1.0, label=f"média ± EP ({fonte_erro})")
ax.axhline(limite, color=COR_PICOS, ls=":", lw=0.8, label=f"triagem (μ + {MULTIPLICADOR_DESVIO}σ)")
ax.scatter(x[picos], y[picos], color=COR_PICOS, s=12, zorder=5)
# picos vizinhos (< SEPARACAO_ROTULOS resíduos): o da esquerda abre para a esquerda e o da direita, para a direita
lados = {}
for a, b in zip(picos[:-1], picos[1:]):
    if x[b] - x[a] < SEPARACAO_ROTULOS:
        lados[a], lados[b] = "right", "left"
for i, texto in zip(picos, rotulos_picos):
    lado = lados.get(i, "center")
    ax.annotate(texto, (x[i], y[i]), xytext=({"right": -3, "left": 3, "center": 0}[lado], 5),
                textcoords="offset points", ha=lado, fontsize=TAMANHO_FONTE - 2, fontweight="bold")
ax.set(xlabel="Resíduo", ylabel="RMSF C$_α$ (nm)", xlim=(x.min(), x.max()), ylim=LIMITES_Y or (0, 1.6 * y.max()))
ax.legend(loc="upper left", ncol=4, frameon=True, fontsize=TAMANHO_FONTE - 2)
fig_perfil = fig
plt.show()

### Figura 2 — comparação com o fator B

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 6.3, 3.25
COR_DM, COR_CRISTAL = "#003f5c", "#bc5090"

# ── Gráfico ─────────────────────────────────────────────────────────────────
fig_bfator = None
if not df_comparacao.empty:
    fig, (ax_p, ax_d) = plt.subplots(1, 2, figsize=(LARGURA, ALTURA), layout="constrained",
                                     gridspec_kw={"width_ratios": [2.4, 1]})
    r = df_correlacao.iloc[0]
    # mesmo eixo: B da DM multiplicado pelo fator de escala de mínimos quadrados (compara a FORMA)
    ax_p.plot(df_comparacao["residuo"], df_comparacao["b_cristal_A2"], color=COR_CRISTAL, lw=0.8, label="cristal 1XNB")
    ax_p.plot(df_comparacao["residuo"], r["escala_exp_sobre_dm"] * df_comparacao["b_dm_A2"], color=COR_DM, lw=0.8,
              label=f"DM × {r['escala_exp_sobre_dm']:.2f} (escala de mínimos quadrados)")
    ax_p.set(xlabel="Resíduo", ylabel="B (Å$^2$)", xlim=(df_comparacao["residuo"].min(), df_comparacao["residuo"].max()))
    ax_p.legend(frameon=True, fontsize=TAMANHO_FONTE - 2, loc="upper left")
    ax_d.scatter(df_comparacao["b_dm_A2"], df_comparacao["b_cristal_A2"], s=8, color=COR_DM, alpha=0.6, lw=0)
    grade = np.linspace(0, df_comparacao["b_dm_A2"].max() * 1.05, 50)
    ax_d.plot(grade, r["escala_exp_sobre_dm"] * grade, color=COR_MEDIA, ls="--", lw=0.8)
    ax_d.plot(grade, r["intercepto_A2"] + r["inclinacao"] * grade, color=COR_CRISTAL, ls=":", lw=0.9)
    ax_d.set(xlabel="B DM (Å$^2$)", ylabel="B cristal (Å$^2$)")
    ax_d.text(0.04, 0.96, f"r = {r['pearson_r']:.2f}   ρ = {r['spearman_rho']:.2f}\n"
                          f"– –  B = {r['escala_exp_sobre_dm']:.2f}·B$_{{DM}}$\n"
                          f"···  B = {r['intercepto_A2']:.1f} + {r['inclinacao']:.2f}·B$_{{DM}}$",
              transform=ax_d.transAxes, va="top", fontsize=TAMANHO_FONTE - 2)
    for eixo, letra in zip((ax_p, ax_d), "AB"):
        eixo.set_title(letra, loc="left", fontweight="bold")
    fig_bfator = fig
    plt.show()

## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_PX = 480                # pixels
LARGURA_PX = None              # None = largura da página
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
# legenda: só as réplicas; a média (linha escura), o IC e o DSSP aparecem no hover
x = df_perfil["residuo"].to_numpy()
tem_dssp = "dssp" in df_perfil
fig_i = make_subplots(rows=2 if tem_dssp else 1, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                      row_heights=[0.9, 0.1] if tem_dssp else [1.0])
fig_i.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[df_perfil["ic_superior_nm"], df_perfil["ic_inferior_nm"][::-1]],
                           fill="toself", fillcolor="rgba(51,51,51,0.15)", line=dict(width=0), hoverinfo="skip",
                           showlegend=False), row=1, col=1)
for i, (rep, g) in enumerate(df_rmsf.groupby("replica", sort=False)):
    fig_i.add_trace(go.Scatter(x=g["residuo"], y=g["rmsf_nm"], mode="lines", name=rep,
                               line=dict(color=CORES[i % len(CORES)], width=1)), row=1, col=1)
fig_i.add_trace(go.Scatter(x=x, y=df_perfil["rmsf_medio_nm"], mode="lines", name="média", showlegend=False,
                           line=dict(color=COR_MEDIA, width=2),
                           customdata=np.c_[df_perfil["nome"], df_perfil["sem_nm"], df_perfil["b_dm_A2"]],
                           hovertemplate="média — %{customdata[0]}%{x}: %{y:.3f} ± %{customdata[1]:.3f} nm"
                                         "<br>B DM = %{customdata[2]:.1f} Å²<extra></extra>"), row=1, col=1)
fig_i.add_annotation(x=0, y=1, xref="x domain", yref="y domain", xanchor="left", yanchor="top", showarrow=False,
                     text=f"linha escura: média ± IC {NIVEL_CONFIANCA:.0%} ({fonte_erro})", row=1, col=1)
if tem_dssp:
    fig_i.add_trace(go.Heatmap(x=x, z=[df_perfil["dssp"].map({"C": 0, "E": 1, "H": 2}).fillna(0)],
                               text=[df_perfil["dssp"]], showscale=False, zmin=0, zmax=2,
                               colorscale=[[0, "#ffffff"], [0.33, "#ffffff"], [0.33, "#f6bd16"], [0.66, "#f6bd16"],
                                           [0.66, "#5b8ff9"], [1, "#5b8ff9"]],
                               hovertemplate="resíduo %{x}: %{text}<extra></extra>", name="DSSP"), row=2, col=1)
    fig_i.update_yaxes(showticklabels=False, title_text="SS", row=2, col=1)
fig_i.update_yaxes(title_text="RMSF Cα (nm)", rangemode="tozero", row=1, col=1)
fig_i.update_xaxes(title_text="Resíduo", row=2 if tem_dssp else 1, col=1)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_PX, width=LARGURA_PX, legend_title_text="Réplica",
                    font=dict(family=FONTE, size=TAMANHO_FONTE_PX))
fig_interativo = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `perfil` | RMSF médio ± SEM e IC por resíduo, B da DM, frações de DSSP |
| `por_replica` | RMSF e MSF de cada réplica |
| `por_bloco` | RMSF por bloco da produção |
| `estatistica` | RMSF médio da proteína por réplica e `entre_replicas` |
| `resumo` | fonte do erro, σ entre resíduos, SEM mediano, picos da triagem |
| `fator_b` | resíduos casados com o PDB: B da DM e B cristalográfico |
| `correlacao_fator_b` | Pearson, Spearman e fator de escala |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"rmsf_perfil": fig_perfil, **({"rmsf_fator_b": fig_bfator} if fig_bfator else {})}
INTERATIVOS = {"rmsf_perfil": fig_interativo}
TABELAS = {"perfil": df_perfil, "por_replica": df_rmsf, "por_bloco": df_blocos, "estatistica": df_stats,
          "resumo": df_resumo, "fator_b": df_comparacao, "correlacao_fator_b": df_correlacao}
PARAMETROS = {"descarte_ns": DESCARTE_NS, "dt_analise_ps": DT_ANALISE_PS, "selecao_rmsf": SELECAO_RMSF,
              "iteracoes_media": ITERACOES_MEDIA, "pdb_bfator": PDB_BFATOR, "deslocamento": DESLOCAMENTO_RESIDUO}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados